In [24]:

import pandas as pd
import os
import probabilistic_automaton as pa
import sampling 

import itertools
from pyranges.readers import read_gtf
from processing_fasta import *
from collections import defaultdict

from Bio import SeqIO

In [5]:

gtf_file_path = '/data/gencode.v49lift37.basic.annotation_protein_coding.gtf' # to build on the automata
cwd = Path(os.getcwd())
print("Loading GTF database into memory, please wait...")
gtf = read_gtf(str(cwd) + gtf_file_path, as_df=True)
print("GTF loaded successfully!")

Loading GTF database into memory, please wait...
GTF loaded successfully!


In [3]:

#Getting the number of genes the 22 chromosomes and the X and Y chromosomes
chr_list = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9',
            'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX', 'chrY']
for chr in chr_list:
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    print(f'number of genes on {chr}: ', number_genes)
    


number of genes on chr1:  2098
number of genes on chr2:  1257
number of genes on chr3:  1085
number of genes on chr4:  762
number of genes on chr5:  897
number of genes on chr6:  1049
number of genes on chr7:  937
number of genes on chr8:  710
number of genes on chr9:  797
number of genes on chr10:  743
number of genes on chr11:  1318
number of genes on chr12:  1042
number of genes on chr13:  321
number of genes on chr14:  619
number of genes on chr15:  607
number of genes on chr16:  860
number of genes on chr17:  1190
number of genes on chr18:  267
number of genes on chr19:  1483
number of genes on chr20:  549
number of genes on chr21:  230
number of genes on chr22:  449
number of genes on chrX:  876
number of genes on chrY:  63


This code takes in: list of chromosome names 
what it does : Creates a folder named "generated_sequences" 
               Creates a folder for each Chromosome in "generated_sequences" 
               For each Chromosome : gets the number of its genes 
                                    For each gene: creates an automata for that gene 
                                                   generates N samples for that gene 
                                                   saves each sample in fasta file in the chromosome's folder
Return: CSV file with the following structure:sample_id, chromosome, gene_location, sequence_length, acceptance_status, acceptance_score, file_path

Test1: Running the code only on Chromosome 22
       The chromosome has: 449 genes
       Trying to Generate N = 100 samples for each gene
       ((( 449 automata and 44 900 generation )))
Results: more than 30 minutes of running 

Test2: Running the code only on Chromosome 22
       The chromosome has: 449 genes BUT WE SELECT THE FIRST 10 GENES 
       Trying to Generate N = 100 samples for each gene
Results: 3 minutes of running. Samples well Generated

Solution? Try MultiProcessing not sure there are no matrix

In [ ]:
database_rows = [] #CSV file 
chr_list = ['chr22'] # for the moment running only on chr22, but we can run it on all chromosomes later

for chr in chr_list:
    os.makedirs(f"generated_sequences/{chr}", exist_ok=True) #folder for each chromosome
    
    #Getting the number of genes on this chromosome
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    
    #Generating 100 samples for each gene on the chromosome
    for location in range(1, number_genes + 1): 
        my_automata = pa.automata_builder(gtf_file_path, chr, gene_number = location) # creating the automata
        N = 5 # number of samples to generate per gene
        for i in range(N):
            sample = sampling.mutated_sample(id = i, chromosome = chr, location = location, sequence = [], automata = my_automata)
            sample.generate_mutated_sample()
            acceptance_status, score = my_automata.accepts(sample.sequence)

            # create a text file for the generated sequence
            file_name = f"{location}_sample_{i}.fasta"
            file_path = f"generated_sequences/{chr}/{file_name}"
            # save the generated sequence in a fasta file
            with open(file_path, "w") as f:
                f.write(f">{location}_sample_{i}\n") # Standard FASTA header
                f.write(sample.sequence)
            
            # save the metadata AND the file path to our database list
            database_rows.append({
                "sample_id": i,
                "chromosome": chr,
                "gene_location": location, 
                "sequence_length": len(sample.sequence),
                "acceptance_status": acceptance_status,
                "acceptance_score": score,
                "file_path": f"{chr}/{file_name}"
            })

            # export the clean, readable database to a CSV so we can use it after 
            df = pd.DataFrame(database_rows)
            df.to_csv("mutated_samples_database.csv", index=False)
            
print(f"The dataset for th {chr_list} is ready!")



### Focused Sampling : 1 gene , 5 000 samples
New Approche: Test 3: Choosing one single gene from the Chromosome 
                      Generating N = 1000 sample for that gene 
Inspiration : To run the CNN we need more than N = 100 sample for one single gene.

In [30]:

database_rows = [] #CSV file 
chr_list = ['chr22'] # for the moment running only on chr22, but we can run it on all chromosomes later
N = 5000 # number of samples to generate per gene

for chr in chr_list:
    os.makedirs(f"focused_sampling/{chr}", exist_ok=True) #folder for each chromosome
    
    #Getting the number of genes on this chromosome
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    
    ### Choosing a specific gene to focus on, for example gene number 10
    focused_gene_number = 10
    location = focused_gene_number
    
    #Generating 100 samples for each gene on the chromosome
    
    my_automata = pa.automata_builder(gtf_file_path, chr, gene_number = location) # creating the automata
    
    for i in range(N):
        sample = sampling.mutated_sample(id = i, chromosome = chr, location = location, sequence = [], automata = my_automata)
        sample.generate_mutated_sample()
        acceptance_status, score = my_automata.accepts(sample.sequence)

                # create a text file for the generated sequence
        file_name = f"{location}_sample_{i}.fasta"
        file_path = f"focused_sampling/{chr}/{file_name}"
                # save the generated sequence in a fasta file
        with open(file_path, "w") as f:
                f.write(f">{location}_sample_{i}\n") # Standard FASTA header
                f.write(sample.sequence)
                
        # save the metadata AND the file path to our database list
        database_rows.append({
                        "sample_id": i,
                        "chromosome": chr,
                        "gene_location": location, 
                        "sequence_length": len(sample.sequence),
                        "acceptance_status": acceptance_status,
                        "acceptance_score": score,
                        "file_path": f"{chr}/{file_name}"
                })

    # export the clean, readable database to a CSV so we can use it after 
    df = pd.DataFrame(database_rows)
    ## named manually 
    df.to_csv("focused_sampling_chr22_g10.csv", index=False)
            
print(f"The dataset for th {chr_list} gene number {location} is ready!")


The dataset for th ['chr22'] gene number 10 is ready!


In [31]:
test3 = pd.read_csv("mutated_samples_database.csv")  
print(test3.head())

   sample_id chromosome  gene_location  sequence_length  acceptance_status  \
0          0      chr22             10             5999               True   
1          1      chr22             10            16272               True   
2          2      chr22             10            10718               True   
3          3      chr22             10            21867               True   
4          4      chr22             10              499               True   

   acceptance_score                file_path  
0       -906.322373  chr22/10_sample_0.fasta  
1      -2302.315596  chr22/10_sample_1.fasta  
2      -1552.326170  chr22/10_sample_2.fasta  
3      -3197.410436  chr22/10_sample_3.fasta  
4        -70.761060  chr22/10_sample_4.fasta  


In [32]:
# Check different lengths of the generated samples
test3["sequence_length"].value_counts()

sequence_length
34       67
947       2
15340     2
8720      2
2269      2
         ..
3282      1
19145     1
37465     1
26886     1
4979      1
Name: count, Length: 924, dtype: int64

### Quality Check of the samples 
- 67 samples with the same length (34)  
- We check that we dont have the same sample multiple times. It can obviously happen.
- In order to compare fasta files we use the Biopython Library with implemented fonctionalities to deal with .fasta files.
#### { "sequences" : ["file1.fasta", "file2.fasta"] }

In [33]:

filtered = test3.query("sequence_length == 34")
print(filtered)
# check  if all the fasta files ar different.
files_by_sequence = {} 

for file_name in filtered["file_path"]:
    
    # we join the folder name with the file_name : "./focused_sampling/nom_fichier.fasta"
    complete_path = os.path.join("./focused_sampling", file_name)
    
    try:
        
        # putting the sequence in a tuple and assigning it as a key to the dictionary 
        file_seq_i = tuple(str(record.seq).upper() for record in SeqIO.parse(complete_path, "fasta"))
        
        if file_seq_i in files_by_sequence:
            # if we alr saw this sequence we add its file_name to the list
            files_by_sequence[file_seq_i].append(file_name)
        else:
            # its the first time that we see this file name
            files_by_sequence[file_seq_i]= [file_name]
            
    except FileNotFoundError:
        print(f"file not found : {complete_path}")

#---------------------------Results--------------------------
print("\n--- Diagnostic ---")
print(f" The length of the dictionary is {len(files_by_sequence)}")
uniques_only = True

for sequence, list in files_by_sequence.items():
    if len(list) > 1:
        tous_uniques = False
        print(f" These {len(list)} files are exactly the same")
        print(f"   -> {', '.join(list)}")

if tous_uniques:
    print("The files are all unique")    

     sample_id chromosome  gene_location  sequence_length  acceptance_status  \
23          23      chr22             10               34               True   
34          34      chr22             10               34               True   
38          38      chr22             10               34               True   
63          63      chr22             10               34               True   
72          72      chr22             10               34               True   
..         ...        ...            ...              ...                ...   
940        940      chr22             10               34               True   
949        949      chr22             10               34               True   
970        970      chr22             10               34               True   
974        974      chr22             10               34               True   
984        984      chr22             10               34               True   

     acceptance_score                  